# Workshop - LVLM-HTR

This notebook is a practical workshop flow for evaluating LVLMs on handwritten text recognition (HTR).

> **How to use this notebook**
> - Run top-to-bottom once.
> - Search for `ADJUST:` comments and replace values for your own setup.
> - Some sections depend on assets (Hugging Face repos, API keys, Label Studio) that you need to configure.

## 1) Setup

We use:
- `htrflow` (pipeline side)
- `vlm-eval` (evaluation side)
- Hugging Face Hub for test data and model outputs

In [ ]:
# ADJUST: set to False if you manage dependencies outside notebook (conda/poetry/etc.)
INSTALL_PACKAGES = True

if INSTALL_PACKAGES:
    %pip install -q -U pip
    %pip install -q htrflow vlm-eval huggingface_hub datasets pandas lxml matplotlib seaborn pyyaml

In [ ]:
import json
import os
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lxml import etree

from huggingface_hub import snapshot_download, HfApi

sns.set_theme(style="whitegrid")

## 2) Introduction

### LVLM HTR introduction
- We want to compare LVLM transcription quality on historical handwritten material.
- Ground truth comes from PAGE XML (`TextEquiv/Unicode`) aligned with page images.
- We score results quantitatively (BoW metrics) and qualitatively (error typing in Label Studio).

### BoW metric (from `vlm-eval`)
- **Precision**: overlap / predicted tokens
- **Recall**: overlap / ground-truth tokens
- **F1**: harmonic mean of precision and recall

### Why qualitative analysis matters
At large scale, single metric numbers hide systematic errors (e.g., names, abbreviations, line breaks, layout issues).

## 3) Configure workshop inputs

In [ ]:
# ADJUST: Hugging Face dataset repo containing page images + PAGE XMLs.
HF_TESTSUITE_REPO = "org-or-user/your-testsuite-repo"

# ADJUST: Optional Hugging Face repo containing pipeline predictions as JSON.
# Example intent: upload your HTRflow outputs and evaluate/inspect them later.
HF_PIPELINE_OUTPUT_REPO = "org-or-user/your-pipeline-outputs"  # can be None

# ADJUST: branch/revision if needed
HF_REVISION = "main"

# ADJUST: local working folder
WORKDIR = Path("workshop_data")
WORKDIR.mkdir(parents=True, exist_ok=True)

print("WORKDIR:", WORKDIR.resolve())

## 4) Download the data and inspect it

In [ ]:
# ADJUST: if repo is private, run `huggingface-cli login` in terminal,
# or set HUGGINGFACE_HUB_TOKEN in env before opening notebook.

testsuite_dir = Path(snapshot_download(
    repo_id=HF_TESTSUITE_REPO,
    repo_type="dataset",  # ADJUST to "model" if your testsuite is in a model repo
    revision=HF_REVISION,
    local_dir=WORKDIR / "testsuite",
    local_dir_use_symlinks=False,
))
print("Downloaded testsuite to:", testsuite_dir)

In [ ]:
# Expected vlm-eval layout example:
# testsuite/
#   archive_1/
#     page_0001.jpg
#     page/
#       page_0001.xml

image_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".jp2"}
image_files = [p for p in testsuite_dir.rglob("*") if p.suffix.lower() in image_exts]
xml_files = list(testsuite_dir.rglob("*.xml"))

print(f"Images: {len(image_files)}")
print(f"XMLs:   {len(xml_files)}")
print("Sample image:", image_files[0] if image_files else "NONE")
print("Sample xml:  ", xml_files[0] if xml_files else "NONE")

In [ ]:
# Parse PAGE XML Unicode counts quickly

def extract_unicode_nodes(xml_path: Path):
    root = etree.parse(str(xml_path))
    return root.xpath("//*[local-name()='TextEquiv']/*[local-name()='Unicode']/text()")

stats = []
for x in xml_files[:2000]:  # ADJUST upper bound for very large sets
    try:
        lines = extract_unicode_nodes(x)
        txt = "\n".join(lines)
        stats.append({
            "xml": str(x.relative_to(testsuite_dir)),
            "line_count": len(lines),
            "char_count": len(txt),
            "token_count": len(txt.split()),
        })
    except Exception as e:
        stats.append({"xml": str(x.relative_to(testsuite_dir)), "error": str(e)})

df_stats = pd.DataFrame(stats)
df_stats.head()

In [ ]:
if not df_stats.empty and "token_count" in df_stats:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df_stats["token_count"].dropna(), bins=40, ax=ax)
    ax.set_title("Token count per PAGE XML")
    plt.show()

## 5) Running pipeline (beforehand)

This workshop assumes you run your HTR pipeline beforehand (e.g., with HTRflow), then upload output JSONs to Hugging Face.

> `ADJUST:` Replace command and paths with your actual HTRflow pipeline.

In [ ]:
# Example shell cell for local pipeline execution (edit before running)
# !htrflow run \
#     --input-dir {testsuite_dir} \
#     --output-dir {WORKDIR / 'pipeline_outputs_local'} \
#     --config path/to/htrflow_config.yaml

print("Edit the cell above with your HTRflow command.")

In [ ]:
# Optional: pull pipeline outputs from Hugging Face if already uploaded
pipeline_outputs_dir = None
if HF_PIPELINE_OUTPUT_REPO:
    pipeline_outputs_dir = Path(snapshot_download(
        repo_id=HF_PIPELINE_OUTPUT_REPO,
        repo_type="dataset",  # ADJUST if needed
        revision=HF_REVISION,
        local_dir=WORKDIR / "pipeline_outputs",
        local_dir_use_symlinks=False,
    ))
    print("Pipeline outputs downloaded to:", pipeline_outputs_dir)
else:
    print("HF_PIPELINE_OUTPUT_REPO is None -> skipping download")

## 6) Running LVLM-eval (Gemini)

In [ ]:
# ADJUST: put your Gemini key in environment before running.
# os.environ["GEMINI_API_KEY"] = "..."

config_path = WORKDIR / "config_gemini.yaml"
config_text = f"""
dataset_root: {testsuite_dir}
output_csv: {WORKDIR / 'results_gemini.csv'}
models:
  - name: gemini-2.5-pro
    provider: gemini
    model: gemini-2.5-pro
    api_key_env: GEMINI_API_KEY
""".strip()

config_path.write_text(config_text, encoding="utf-8")
print(config_path.read_text())

In [ ]:
# Run evaluation with CLI
# ADJUST: model name and rate limits as needed
!vlm-eval run --config {config_path}

In [ ]:
results_csv = WORKDIR / "results_gemini_gemini-2.5-pro.csv"  # naming done by vlm-eval
if not results_csv.exists():
    # fallback if output naming differs
    candidates = sorted(WORKDIR.glob("results_gemini*.csv"))
    results_csv = candidates[0] if candidates else None

if results_csv and Path(results_csv).exists():
    df_res = pd.read_csv(results_csv)
    display(df_res.head())
    display(df_res[df_res["level"] == "entire_testset"])
else:
    print("Could not find output CSV. ADJUST filename logic.")

## 7) Setting up Label Studio for qualitative analysis

In [ ]:
# Install + start (terminal alternatives shown as notebook shell lines)
# !pip install -q label-studio
# !label-studio start

print("Open Label Studio in browser, create a project, and import model outputs + references.")

Suggested annotation schema (error typing):
- Substitution (wrong word)
- Omission (missing content)
- Insertion (extra content)
- Named entity issue (person/place)
- Abbreviation expansion error
- Layout/reading-order error
- Uncertain handwriting handling

**Line-by-line vs page-by-page?**
- **Line-by-line**: better for precise error taxonomy and inter-annotator agreement.
- **Page-by-page**: better for workflow speed and layout-related observations.

Recommendation: start line-by-line for a stratified sample, then page-by-page for throughput.

## 8) Qualitative analysis worksheet

In [ ]:
# Template table to fill during manual review
qual_cols = [
    "sample_id", "model", "granularity", "error_type", "severity",
    "comment", "gt_text", "pred_text"
]
qual_df = pd.DataFrame(columns=qual_cols)
qual_df.head()

In [ ]:
# Example aggregation after you annotate
if not qual_df.empty:
    pivot = pd.crosstab(qual_df["model"], qual_df["error_type"])
    display(pivot)
    pivot.plot(kind="bar", stacked=True, figsize=(10, 4))
    plt.title("Error type distribution by model")
    plt.ylabel("Count")
    plt.show()
else:
    print("Populate qual_df first (or load exported annotations).")

## 9) Discussion prompts

Use these in the workshop debrief:

1. **Differences in types of errors**
   - Which models over/under-predict tokens?
   - Which models fail on names/abbreviations?
   - Are layout errors clustered by page complexity?

2. **Pros and cons per method**
   - Image-only transcription vs correction of existing OCR/HTR.
   - Prompting with/without transcription guidelines.
   - Cost/latency vs quality tradeoffs.

3. **What to adjust next**
   - Prompt design
   - Model selection
   - Pre-segmentation quality
   - Post-correction strategy

## 10) Checklist of placeholders to adjust (`ADJUST:`)

- `HF_TESTSUITE_REPO`
- `HF_PIPELINE_OUTPUT_REPO`
- `repo_type` for both snapshot downloads
- dependency install policy (`INSTALL_PACKAGES`)
- HTRflow command and config
- Gemini model + API key env
- output CSV filename fallback logic
- Label Studio import/export wiring